In [1]:
import random

# ==========================================
# PARÁMETROS DE ENTRADA CONFIGURABLES
# ==========================================
# a) Longitud de la cadena de bits
num_bits = 5 

# b) Espacio o rango de búsqueda (min, max)
search_space = (0, 31) 

# c) Función de aptitud (fitness function)
# Usaremos f(x) = x^2 como ejemplo clásico
def fitness_function(x):
    return x**2

# d) Probabilidad de cruza y mutación
p_crossover = 0.8  # 80% de probabilidad de cruza
p_mutation = 0.1   # 10% de probabilidad de mutación por bit

# Otros parámetros del algoritmo
pop_size = 4       # Tamaño de la población
generations = 10   # Número de generaciones

# ==========================================
# FUNCIONES AUXILIARES
# ==========================================

def decode(binary_str, space):
    """
    Decodifica una cadena binaria a un valor decimal dentro del espacio de búsqueda.
    """
    # Convertimos la cadena binaria a entero
    integer_val = int(binary_str, 2)
    # Escalamos el valor al rango de búsqueda deseado
    max_int = (2**len(binary_str)) - 1
    min_val, max_val = space
    decoded_val = min_val + (integer_val / max_int) * (max_val - min_val)
    return decoded_val

def initialize_population(size, bits):
    """
    Paso 1: Inicializa la población con cadenas binarias aleatorias.
    """
    return [''.join(random.choice(['0', '1']) for _ in range(bits)) for _ in range(size)]

# ==========================================
# OPERADORES GENÉTICOS
# ==========================================

def selection(population, fitness_scores):
    """
    Paso 2: Selección (Método de la Ruleta).
    Selecciona individuos basándose en su aptitud relativa.
    """
    total_fitness = sum(fitness_scores)
    # Probabilidades de selección para cada individuo
    probabilities = [f / total_fitness for f in fitness_scores]
    
    # Seleccionamos 'pop_size' individuos usando las probabilidades
    selected = random.choices(population, weights=probabilities, k=len(population))
    return selected

def crossover(parent1, parent2, prob):
    """
    Paso 3: Cruza (Un punto de corte).
    Intercambio aleatorio de la información genética entre dos padres.
    """
    if random.random() < prob:
        # Elegimos un punto de cruza aleatorio
        pt = random.randint(1, len(parent1) - 1)
        # Generamos los hijos combinando las partes
        child1 = parent1[:pt] + parent2[pt:]
        child2 = parent2[:pt] + parent1[pt:]
        return child1, child2
    return parent1, parent2

def mutate(individual, prob):
    """
    Paso 4: Mutación.
    Alteración aleatoria de los bits dentro del cromosoma.
    """
    mutated = ""
    for bit in individual:
        if random.random() < prob:
            # Voltea el bit de 0 a 1 o de 1 a 0
            mutated += '1' if bit == '0' else '0'
        else:
            mutated += bit
    return mutated

# ==========================================
# CICLO PRINCIPAL DEL ALGORITMO
# ==========================================

def simple_genetic_algorithm():
    print("--- INICIANDO ALGORITMO GENÉTICO SIMPLE ---")
    # 1. Inicializar población
    population = initialize_population(pop_size, num_bits)
    
    for gen in range(generations):
        # 2. Evaluar la aptitud de cada individuo
        decoded_pop = [decode(ind, search_space) for ind in population]
        fitness_scores = [fitness_function(x) for x in decoded_pop]
        
        # Encontrar al mejor de la generación actual para monitoreo
        best_fitness = max(fitness_scores)
        best_individual = population[fitness_scores.index(best_fitness)]
        print(f"Generación {gen+1} | Mejor Genotipo: {best_individual} | Valor: {decode(best_individual, search_space):.2f} | Fitness: {best_fitness:.2f}")
        
        # 3. Selección
        selected_pop = selection(population, fitness_scores)
        
        # 4. Cruza
        next_generation = []
        for i in range(0, pop_size, 2):
            parent1 = selected_pop[i]
            # Aseguramos que haya un parent2 (por si la población es impar)
            parent2 = selected_pop[i+1] if (i+1) < pop_size else selected_pop[0] 
            
            child1, child2 = crossover(parent1, parent2, p_crossover)
            next_generation.extend([child1, child2])
            
        # 5. Mutación
        next_generation = [mutate(ind, p_mutation) for ind in next_generation]
        
        # Actualizamos la población para el siguiente ciclo
        population = next_generation[:pop_size]

# Ejecutar el algoritmo
simple_genetic_algorithm()

--- INICIANDO ALGORITMO GENÉTICO SIMPLE ---
Generación 1 | Mejor Genotipo: 11100 | Valor: 28.00 | Fitness: 784.00
Generación 2 | Mejor Genotipo: 11100 | Valor: 28.00 | Fitness: 784.00
Generación 3 | Mejor Genotipo: 11101 | Valor: 29.00 | Fitness: 841.00
Generación 4 | Mejor Genotipo: 11101 | Valor: 29.00 | Fitness: 841.00
Generación 5 | Mejor Genotipo: 11101 | Valor: 29.00 | Fitness: 841.00
Generación 6 | Mejor Genotipo: 11111 | Valor: 31.00 | Fitness: 961.00
Generación 7 | Mejor Genotipo: 11111 | Valor: 31.00 | Fitness: 961.00
Generación 8 | Mejor Genotipo: 11111 | Valor: 31.00 | Fitness: 961.00
Generación 9 | Mejor Genotipo: 11111 | Valor: 31.00 | Fitness: 961.00
Generación 10 | Mejor Genotipo: 11111 | Valor: 31.00 | Fitness: 961.00


In [ ]:
import math

# ==========================================
# DEFINICIÓN DEL PROBLEMA
# ==========================================
# Definimos una lista de ciudades con coordenadas (x, y)
ciudades = {
    'A': (0, 0),
    'B': (2, 4),
    'C': (5, 2),
    'D': (7, 6),
    'E': (8, 1)
}

def calcular_distancia(coord1, coord2):
    """
    Calcula la distancia Euclidiana entre dos puntos (ciudades).
    """
    x1, y1 = coord1
    x2, y2 = coord2
    return math.sqrt((x2 - x1)**2 + (y2 - y1)**2)

# ==========================================
# ALGORITMO ÁVIDO (GOLOSO) PARA TSP
# ==========================================

def tsp_avido(ciudades, ciudad_inicio):
    print("--- INICIANDO ALGORITMO ÁVIDO (TSP) ---")
    
    ciudades_pendientes = list(ciudades.keys())
    ruta_optimizada = []
    distancia_total = 0.0
    
    # 1. Establecer la ciudad de inicio
    ciudad_actual = ciudad_inicio
    ruta_optimizada.append(ciudad_actual)
    ciudades_pendientes.remove(ciudad_actual)
    
    print(f"Inicio del recorrido en: {ciudad_actual}")
    
    # 2. Ciclo principal: mientras haya ciudades por visitar
    while ciudades_pendientes:
        distancia_minima = float('inf')
        ciudad_mas_cercana = None
        
        # 3. Buscar la ciudad no visitada más cercana entre las demás
        for ciudad_candidata in ciudades_pendientes:
            dist = calcular_distancia(ciudades[ciudad_actual], ciudades[ciudad_candidata])
            
            # Tomamos la decisión "golosa": nos quedamos con la distancia menor
            if dist < distancia_minima:
                distancia_minima = dist
                ciudad_mas_cercana = ciudad_candidata
                
        # 4. Moverse a la ciudad más cercana encontrada
        ruta_optimizada.append(ciudad_mas_cercana)
        distancia_total += distancia_minima
        ciudades_pendientes.remove(ciudad_mas_cercana)
        
        print(f"Viajando de {ciudad_actual} a {ciudad_mas_cercana} | Distancia: {distancia_minima:.2f}")
        ciudad_actual = ciudad_mas_cercana
        
    # 5. Regresar a la ciudad de origen para cerrar el ciclo
    dist_regreso = calcular_distancia(ciudades[ciudad_actual], ciudades[ciudad_inicio])
    ruta_optimizada.append(ciudad_inicio)
    distancia_total += dist_regreso
    
    print(f"Regresando de {ciudad_actual} a {ciudad_inicio} | Distancia: {dist_regreso:.2f}")
    print("-" * 40)
    print(f"Ruta final sugerida: {' -> '.join(ruta_optimizada)}")
    print(f"Distancia total recorrida: {distancia_total:.2f}")
    
    return ruta_optimizada, distancia_total

# Ejecutar el algoritmo 
# comenzando, por ejemplo, en la ciudad 'A'
tsp_avido(ciudades, 'A')

--- INICIANDO ALGORITMO ÁVIDO (TSP) ---
Inicio del recorrido en: A
Viajando de A a B | Distancia: 4.47
Viajando de B a C | Distancia: 3.61
Viajando de C a E | Distancia: 3.16
Viajando de E a D | Distancia: 5.10
Regresando de D a A | Distancia: 9.22
----------------------------------------
Ruta final sugerida: A -> B -> C -> E -> D -> A
Distancia total recorrida: 25.56


(['A', 'B', 'C', 'E', 'D', 'A'], 25.55852886151762)